# FindMaximumSubarray and FindMaxCrossingSubarray (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/FindMaximumSubarray_cpp.ipynb)

Chapter 17 — Recursive algorithms. Runnable notebook of `FindMaximumSubarray` and `FindMaxCrossingSubarray`: the algorithms in C++, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/FindMaximumSubarray_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/FindMaximumSubarray_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `RecursiveSum` (PascalCase).

## FindMaxCrossingSubarray and FindMaximumSubarray (divide and conquer)

- **FindMaxCrossingSubarray**$(a,l,mid,r)$ — *Input*: an array and three indices $l\le mid<r$. *Output*: the indices $maxleft\le mid<maxright$ of a maximum-sum subarray among those crossing the midpoint, and its sum. It scans the left half from $mid$ down to $l$ and the right half from $mid+1$ to $r$: $\Theta(n)$.
- **FindMaximumSubarray**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the indices $i\le j$ of a maximum-sum subarray of $a[l..r]$ and its sum. The best among the maximum subarray of the left half, of the right half and of those crossing the midpoint (in case of ties, the left one first, then the right one). $S(n)=2S(n/2)+\Theta(n)=\Theta(n\log_2 n)$, $2n-1$ calls.

The $-\infty$ of the pseudo-code is the smallest representable value. First the trace on the example $a=[-2,5,-1,3]$ (like the execution of the C++ implementation of the chapter; result $(1,3,7)$, that is $(2,4,7)$ with indices from 1), then the stock example, with the daily changes $a=[13,-3,-25,20,-3,-16,-23,18,20,-7,12,-5,-22,15,-4,7]$: table of the results of the calls on the sub-intervals (by level of the tree, as in the chapter; indices are from 0) and the computations of the two `FindMaxCrossingSubarray` of level 1. The result is $(7,10,43)$, that is $(8,11,43)$ with indices from 1.

In [1]:
%%writefile maxsub_dc.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int fmsCalls = 0;   // calls of FindMaximumSubarray

struct Triple
{
    int i, j, sum;
};

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

string Tr(const Triple &t)
{
    return "(" + to_string(t.i) + ", " + to_string(t.j) + ", " + to_string(t.sum) + ")";
}

// Returns (maxleft, maxright, sum) of the best subarray crossing mid
Triple FindMaxCrossingSubarray(int *a, int l, int mid, int r)
{
    int leftsum = INT_MIN;                  // -infinity
    int sum = 0;
    int maxleft = mid;
    for (int i = mid; i >= l; i--)          // from mid down to l
    {
        sum = sum + a[i];                   // sum of a[i..mid]
        if (sum > leftsum)
        {
            leftsum = sum;
            maxleft = i;
        }
    }
    int rightsum = INT_MIN;
    sum = 0;
    int maxright = mid + 1;
    for (int j = mid + 1; j <= r; j++)      // from mid+1 to r
    {
        sum = sum + a[j];                   // sum of a[mid+1..j]
        if (sum > rightsum)
        {
            rightsum = sum;
            maxright = j;
        }
    }
    return {maxleft, maxright, leftsum + rightsum};
}

// Returns (i, j, sum) of a maximum-sum subarray of a[l..r]
Triple FindMaximumSubarray(int *a, int l, int r)
{
    fmsCalls++;
    if (l == r)
    {
        return {l, r, a[l]};                // base case: one element
    }
    int mid = l + (r - l) / 2;
    Triple left = FindMaximumSubarray(a, l, mid);
    Triple right = FindMaximumSubarray(a, mid + 1, r);
    Triple cross = FindMaxCrossingSubarray(a, l, mid, r);
    if (left.sum >= right.sum && left.sum >= cross.sum)
    {
        return left;
    }
    if (right.sum >= left.sum && right.sum >= cross.sum)
    {
        return right;
    }
    return cross;
}

// ---- variants that print the trace ----
Triple Best(const Triple &left, const Triple &right, const Triple &cross)
{
    if (left.sum >= right.sum && left.sum >= cross.sum) return left;
    if (right.sum >= left.sum && right.sum >= cross.sum) return right;
    return cross;
}

Triple FmsTrace(int *a, int l, int r, int depth)
{
    string pad(2 * depth, ' ');
    if (l == r)
    {
        printf("%sFindMaximumSubarray(a, %d, %d) = (%d, %d, %d)  base case\n", pad.c_str(), l, r, l, r, a[l]);
        return {l, r, a[l]};
    }
    int mid = l + (r - l) / 2;
    printf("%sFindMaximumSubarray(a, %d, %d): l != r, mid = %d\n", pad.c_str(), l, r, mid);
    Triple left = FmsTrace(a, l, mid, depth + 1);
    Triple right = FmsTrace(a, mid + 1, r, depth + 1);
    Triple cross = FindMaxCrossingSubarray(a, l, mid, r);
    printf("%s  FindMaxCrossingSubarray(a, %d, %d, %d) = %s\n", pad.c_str(), l, mid, r, Tr(cross).c_str());
    Triple ret = Best(left, right, cross);
    printf("%s  left.sum=%d, right.sum=%d, cross.sum=%d => returns %s\n", pad.c_str(), left.sum, right.sum, cross.sum, Tr(ret).c_str());
    return ret;
}

struct Row
{
    int l, r, mid;
    Triple left, right, cross, ret;
};

Triple FmsRows(int *a, int l, int r, vector<Row> &rows)
{
    if (l == r)
    {
        return {l, r, a[l]};
    }
    int mid = l + (r - l) / 2;
    Triple left = FmsRows(a, l, mid, rows);
    Triple right = FmsRows(a, mid + 1, r, rows);
    Triple cross = FindMaxCrossingSubarray(a, l, mid, r);
    Triple ret = Best(left, right, cross);
    rows.push_back({l, r, mid, left, right, cross, ret});
    return ret;
}

void CrossingTrace(int *a, int l, int mid, int r)
{
    printf("FindMaxCrossingSubarray(a, %d, %d, %d)\n", l, mid, r);
    printf("%3s %5s %5s %8s %8s\n", "i", "a[i]", "sum", "leftsum", "maxleft");
    int leftsum = INT_MIN, maxleft = mid, sum = 0;
    for (int i = mid; i >= l; i--)
    {
        sum = sum + a[i];
        if (sum > leftsum)
        {
            leftsum = sum;
            maxleft = i;
        }
        printf("%3d %5d %5d %8d %8d\n", i, a[i], sum, leftsum, maxleft);
    }
    printf("%3s %5s %5s %8s %8s\n", "j", "a[j]", "sum", "rightsum", "maxright");
    int rightsum = INT_MIN, maxright = mid + 1;
    sum = 0;
    for (int j = mid + 1; j <= r; j++)
    {
        sum = sum + a[j];
        if (sum > rightsum)
        {
            rightsum = sum;
            maxright = j;
        }
        printf("%3d %5d %5d %8d %8d\n", j, a[j], sum, rightsum, maxright);
    }
    printf("crossing subarray a[%d..%d], sum %d + %d = %d\n", maxleft, maxright, leftsum, rightsum, leftsum + rightsum);
}

#ifndef NOMAIN
int main()
{
    int a[] = {-2, 5, -1, 3};
    printf("a = %s\n", Fmt(a, 4).c_str());
    Triple res = FmsTrace(a, 0, 3, 0);
    fmsCalls = 0;
    FindMaximumSubarray(a, 0, 3);
    printf("result %s   (%d, %d, %d with indices from 1)   calls=%d (2n-1=%d)\n", Tr(res).c_str(), res.i + 1, res.j + 1, res.sum, fmsCalls, 2 * 4 - 1);

    printf("\n");
    int b[] = {13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7};
    int n = 16;
    printf("a = %s\n", Fmt(b, n).c_str());
    fmsCalls = 0;
    res = FindMaximumSubarray(b, 0, n - 1);
    printf("FindMaximumSubarray(a, 0, %d) = %s   (%d, %d, %d with indices from 1)   calls=%d (2n-1=%d)\n", n - 1, Tr(res).c_str(), res.i + 1, res.j + 1, res.sum, fmsCalls, 2 * n - 1);
    vector<Row> rows;
    FmsRows(b, 0, n - 1, rows);
    sort(rows.begin(), rows.end(), [](const Row &x, const Row &y)
         { return (x.r - x.l) != (y.r - y.l) ? (x.r - x.l) > (y.r - y.l) : x.l < y.l; });
    printf("%-9s %3s  %-13s %-13s %-13s returned\n", "call", "mid", "left", "right", "crossing");
    for (const Row &w : rows)
    {
        string call = "[" + to_string(w.l) + ".." + to_string(w.r) + "]";
        printf("%-9s %3d  %-13s %-13s %-13s %s\n", call.c_str(), w.mid, Tr(w.left).c_str(), Tr(w.right).c_str(), Tr(w.cross).c_str(), Tr(w.ret).c_str());
    }
    printf("\n");
    CrossingTrace(b, 0, 3, 7);
    printf("\n");
    CrossingTrace(b, 8, 11, 15);
    return 0;
}
#endif

Writing maxsub_dc.cpp


In [2]:
!g++ -std=c++17 -O2 -o maxsub_dc maxsub_dc.cpp && ./maxsub_dc

a = [-2, 5, -1, 3]
FindMaximumSubarray(a, 0, 3): l != r, mid = 1
  FindMaximumSubarray(a, 0, 1): l != r, mid = 0
    FindMaximumSubarray(a, 0, 0) = (0, 0, -2)  base case
    FindMaximumSubarray(a, 1, 1) = (1, 1, 5)  base case
    FindMaxCrossingSubarray(a, 0, 0, 1) = (0, 1, 3)
    left.sum=-2, right.sum=5, cross.sum=3 => returns (1, 1, 5)
  FindMaximumSubarray(a, 2, 3): l != r, mid = 2
    FindMaximumSubarray(a, 2, 2) = (2, 2, -1)  base case
    FindMaximumSubarray(a, 3, 3) = (3, 3, 3)  base case
    FindMaxCrossingSubarray(a, 2, 2, 3) = (2, 3, 2)
    left.sum=-1, right.sum=3, cross.sum=2 => returns (3, 3, 3)
  FindMaxCrossingSubarray(a, 0, 1, 3) = (1, 3, 7)
  left.sum=5, right.sum=3, cross.sum=7 => returns (1, 3, 7)
result (1, 3, 7)   (2, 4, 7 with indices from 1)   calls=7 (2n-1=7)

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
FindMaximumSubarray(a, 0, 15) = (7, 10, 43)   (8, 11, 43 with indices from 1)   calls=31 (2n-1=31)
call      mid  left          righ

**Try it yourself.** Change the array: the maximum subarray may have a single element or cross the midpoint; the calls are always $2n-1$.

In [3]:
%%writefile prova_maxsub_dc.cpp
#define NOMAIN
#include "maxsub_dc.cpp"

int main()
{
    // Change the array: how many elements does the maximum subarray have? The calls are always 2n-1.
    int b[] = {4, -1, 2, 1, -5, 4, -7, 3, 3};
    fmsCalls = 0;
    Triple t = FindMaximumSubarray(b, 0, 8);
    printf("%s %d\n", Tr(t).c_str(), fmsCalls);
    return 0;
}

Writing prova_maxsub_dc.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_maxsub_dc prova_maxsub_dc.cpp && ./prova_maxsub_dc

(0, 3, 6) 17


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).